# NC-SRAG lean run (fits ~1.8h): natural noise + 4-bit judge

No 32B (out of budget). Section 1 = natural-noise experiment (Qwen-7B, full pool,
no injection). Section 2 = 4-bit judge over the naturalistic files, natural FIRST,
so if the runtime is reclaimed you still keep the new data and the natural labels.
Everything checkpoints to Drive and resumes.

In [ ]:
!pip -q install transformers accelerate datasets faiss-cpu sentence-transformers rank_bm25 scikit-learn scipy bitsandbytes
from google.colab import drive; drive.mount('/content/drive')
import os

## Section 1 - natural noise (Qwen2.5-7B, full pool, NO injection) ~40 min

In [ ]:
%%writefile harness_natural.py
import os, json, re, math, time, random, gc
import numpy as np, torch, faiss
from datasets import load_dataset
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
WORK="/content/drive/MyDrive/ncsrag_nat"; os.makedirs(WORK, exist_ok=True)
GEN=os.environ.get("PILOT_MODEL","Qwen/Qwen2.5-7B-Instruct")
N_Q=int(os.environ.get("N_Q","300"))
NSAMP=int(os.environ.get("NSAMP","4"))
GTAG=re.sub(r"[^a-zA-Z0-9]+","_",GEN.split("/")[-1]).lower()

ds=load_dataset("rajpurkar/squad", split="validation")
# FULL pool: every unique context paragraph (~2000). Retrieval can genuinely miss
# the gold paragraph, so the noise is whatever real retrieval returns (no injection).
passages=[]; pid_of_ctx={}
for r in ds:
    c=r["context"]
    if c not in pid_of_ctx: pid_of_ctx[c]=len(passages); passages.append(c)
# same 300 questions as the injected run (seed=13, one per paragraph) for comparability
dsh=load_dataset("rajpurkar/squad", split="validation").shuffle(seed=13)
queries=[]; seen=set()
for r in dsh:
    c=r["context"]
    if c in seen: continue
    a=r["answers"]["text"][0] if r["answers"]["text"] else ""
    if not a: continue
    seen.add(c); queries.append({"qid":f"sq{len(queries)}","question":r["question"],"gold":a,"gold_pid":pid_of_ctx[c]})
    if len(queries)>=N_Q: break
print("FULL pool",len(passages),"queries",len(queries),flush=True)

tok_corpus=[re.findall(r"\w+",p.lower()) for p in passages]
bm25=BM25Okapi(tok_corpus)
def bm25_top(q,k=4):
    sc=bm25.get_scores(re.findall(r"\w+",q.lower())); idx=list(np.argsort(-sc)[:k]); return idx,[float(sc[i]) for i in idx]
enc_model=None; index=None
def build_dense():
    global enc_model,index
    enc_model=SentenceTransformer("intfloat/e5-base-v2",device="cuda")
    emb=enc_model.encode(["passage: "+p for p in passages],batch_size=128,
                         show_progress_bar=True,normalize_embeddings=True).astype(np.float32)
    index=faiss.IndexFlatIP(emb.shape[1]); index.add(emb)
def dense_top(q,k=4):
    qv=enc_model.encode(["query: "+q],normalize_embeddings=True).astype(np.float32)
    s,idx=index.search(qv,k); return list(idx[0]),[float(x) for x in s[0]]

def norm(s): return re.sub(r"[^a-z0-9 ]","",s.lower()).strip()
ABST=re.compile(r"(don.?t know|unknown|cannot|not (mentioned|stated|provided|given)|no answer)")
def label(a,g):
    a,g=norm(a),norm(g)
    if not a or ABST.search(a): return "abstain"
    if g and (g in a or a in g): return "correct"
    return "hallucination"
def make_prompt(tok,sysmsg,user):
    try: return tok.apply_chat_template([{"role":"system","content":sysmsg},{"role":"user","content":user}],tokenize=False,add_generation_prompt=True)
    except Exception: return tok.apply_chat_template([{"role":"user","content":sysmsg+"\n\n"+user}],tokenize=False,add_generation_prompt=True)
SYS="Answer with the shortest factual answer using the passages. If the passages do not contain the answer, answer exactly: unknown"

CONDS=["closedbook","retrieved"]   # NO injection: natural retrieval noise only
def run(retriever):
    TAG=f"natural_{retriever}_{GTAG}"
    tok=AutoTokenizer.from_pretrained(GEN)
    model=AutoModelForCausalLM.from_pretrained(GEN,torch_dtype=torch.bfloat16,device_map="auto").eval()
    if retriever=="dense" and index is None: build_dense()
    topfn=bm25_top if retriever=="bm25" else dense_top
    @torch.no_grad()
    def gen(p,sample=False,n=1):
        enc=tok(p,return_tensors="pt",truncation=True,max_length=2048).to(model.device)
        out=model.generate(**enc,max_new_tokens=24,do_sample=sample,temperature=0.7 if sample else None,
            top_p=0.9 if sample else None,num_return_sequences=n,output_scores=True,
            return_dict_in_generate=True,pad_token_id=tok.eos_token_id)
        seqs=out.sequences[:,enc["input_ids"].shape[1]:]
        txt=[tok.decode(s,skip_special_tokens=True).strip() for s in seqs]
        lps=[];ents=[]
        for step,scr in enumerate(out.scores):
            lp=torch.log_softmax(scr[0].float(),-1); tid=seqs[0][step] if step<seqs.shape[1] else seqs[0][-1]
            lps.append(float(lp[tid])); pr=lp.exp(); ents.append(float(-(pr*lp).sum()))
        return txt,(lps or [0.0]),(ents or [0.0])
    def se_disc(samples):
        from collections import Counter
        c=Counter(norm(s) for s in samples); tot=sum(c.values())
        return float(-sum((v/tot)*math.log(v/tot) for v in c.values()))
    part=f"{WORK}/results_{TAG}_partial.json"
    rows=json.load(open(part)) if os.path.exists(part) else []
    done={(r["qid"],r["cond"]) for r in rows}; t0=time.time()
    for qi,q in enumerate(queries):
        for cond in CONDS:
            if (q["qid"],cond) in done: continue
            if cond=="closedbook":
                ctx,sc,gold_in=[],[],0.0
            else:
                idx,sc=topfn(q["question"],4); ctx=[passages[i] for i in idx]
                gold_in=1.0 if q["gold_pid"] in idx else 0.0
            p=make_prompt(tok,SYS,("Passages:\n"+"\n".join("- "+t for t in ctx)+"\n\n" if ctx else "")+f"Question: {q['question']}")
            txt,lps,ents=gen(p); g=txt[0]; lab=label(g,q["gold"])
            samples,_,_=gen(p,sample=True,n=NSAMP); se=se_disc(samples)
            if ctx:
                cs=sorted([float(x) for x in sc],reverse=True) or [0.0,0.0]
                while len(cs)<2: cs.append(0.0)
                feat=dict(ret_mean=float(np.mean(cs)),ret_top1=cs[0],ret_margin=cs[0]-cs[1],ret_min=cs[-1],ret_std=float(np.std(cs)),has_ctx=1.0)
            else: feat=dict(ret_mean=0,ret_top1=0,ret_margin=0,ret_min=0,ret_std=0,has_ctx=0.0)
            feat.update(lp_mean=float(np.mean(lps)),lp_min=float(np.min(lps)),ent_mean=float(np.mean(ents)),ent_max=float(np.max(ents)),ent_first=ents[0],ans_len=len(lps),se=se,gold_in_ctx=gold_in)
            rows.append(dict(qid=q["qid"],regime="natural",cond=cond,question=q["question"],gold=q["gold"],answer=g,label=lab,**feat))
            done.add((q["qid"],cond))
        if qi%5==0:
            print(f"  {TAG}: q {qi}/{len(queries)} rows={len(rows)} t={time.time()-t0:.0f}s",flush=True)
            json.dump(rows,open(part,"w"))
    json.dump(rows,open(f"{WORK}/results_{TAG}.json","w"))
    print("DONE",TAG,len(rows),flush=True)
    del model; gc.collect(); torch.cuda.empty_cache()


In [ ]:
import os
os.environ["PILOT_MODEL"]="Qwen/Qwen2.5-7B-Instruct"
os.environ["N_Q"]="300"
import importlib, harness_natural; importlib.reload(harness_natural)
for retr in ["bm25","dense"]:
    print("\n=== NATURAL", retr, "===", flush=True)
    try: harness_natural.run(retr)
    except Exception as e:
        import gc, torch; print("SKIPPED", retr, "->", repr(e)[:300]); gc.collect(); torch.cuda.empty_cache()
print("\nSECTION 1 DONE: results_natural_*_qwen2_5_7b_instruct.json")

## Section 2 - 4-bit judge (Qwen2.5-14B), natural files first ~40 min
Resumes any existing `judged_*_partial.json`. Fits fully on GPU in 4-bit (no CPU offload).

In [ ]:
%%writefile judge_lean.py
import os, re, json, glob, time
import numpy as np, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
WORK="/content/drive/MyDrive/ncsrag_nat"
JUDGE=os.environ.get("JUDGE_MODEL","Qwen/Qwen2.5-14B-Instruct")
print("judge model:",JUDGE,flush=True)
tok=AutoTokenizer.from_pretrained(JUDGE)
from transformers import BitsAndBytesConfig
_bnb=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",
                        bnb_4bit_compute_dtype=torch.bfloat16,bnb_4bit_use_double_quant=True)
model=AutoModelForCausalLM.from_pretrained(JUDGE,quantization_config=_bnb,device_map="auto").eval()

SYS=("You are a strict grader for short-answer question answering. You are given a "
 "question, a reference answer that is known to be correct, and a candidate answer "
 "produced by another system. Decide whether the candidate is correct. The candidate "
 "is CORRECT if it conveys the same factual answer as the reference, ignoring phrasing, "
 "extra words, capitalisation, or added explanation. It is WRONG if it gives a different "
 "or contradictory answer. It is REFUSED if it declines, says it does not know, or says "
 "the passages do not contain the answer. Reply with exactly one word: CORRECT, WRONG, or REFUSED.")

def make_prompt(q,gold,cand):
    user=f"Question: {q}\nReference answer: {gold}\nCandidate answer: {cand}\nYour one-word verdict:"
    try:
        return tok.apply_chat_template([{"role":"system","content":SYS},{"role":"user","content":user}],
                                       tokenize=False,add_generation_prompt=True)
    except Exception:
        return tok.apply_chat_template([{"role":"user","content":SYS+"\n\n"+user}],
                                       tokenize=False,add_generation_prompt=True)

@torch.no_grad()
def verdict(q,gold,cand):
    enc=tok(make_prompt(q,gold,cand),return_tensors="pt",truncation=True,max_length=2048).to(model.device)
    out=model.generate(**enc,max_new_tokens=4,do_sample=False,pad_token_id=tok.eos_token_id)
    t=tok.decode(out[0][enc["input_ids"].shape[1]:],skip_special_tokens=True).upper()
    if "REFUS" in t or "UNKNOWN" in t: return "abstain"
    if "INCORRECT" in t or "WRONG" in t: return "hallucination"
    if "CORRECT" in t: return "correct"
    return "hallucination"

def kappa(a,b,labels):
    idx={l:i for i,l in enumerate(labels)}; n=len(a); k=len(labels)
    if n==0: return float("nan"), np.zeros((k,k))
    m=np.zeros((k,k))
    for x,y in zip(a,b): m[idx[x],idx[y]]+=1
    po=np.trace(m)/n
    pe=sum((m[i,:].sum()/n)*(m[:,i].sum()/n) for i in range(k))
    return ((po-pe)/(1-pe) if pe<1 else 1.0), m

def run():
    files=sorted(glob.glob(f"{WORK}/results_natural_*.json"))+sorted(glob.glob(f"{WORK}/results_squad_*.json"))
    files=[f for f in files if "_partial" not in f]
    print("files to judge:",[os.path.basename(f) for f in files],flush=True)
    summary={}
    for f in files:
        rows=json.load(open(f)); tag=os.path.basename(f)[len("results_"):-len(".json")]
        part=f"{WORK}/judged_{tag}_partial.json"
        out=json.load(open(part)) if os.path.exists(part) else []
        done={r["qid"]+"_"+r["cond"] for r in out}; t0=time.time()
        for i,r in enumerate(rows):
            if r["qid"]+"_"+r["cond"] in done: continue
            jl=verdict(r["question"],r["gold"],r["answer"])
            nr=dict(r); nr["label_em"]=r["label"]; nr["label"]=jl; out.append(nr)
            if i%200==0:
                print(f"  {tag}: {i}/{len(rows)} t={time.time()-t0:.0f}s",flush=True)
                json.dump(out,open(part,"w"))
        json.dump(out,open(f"{WORK}/judged_{tag}.json","w"))
        em=[r["label_em"] for r in out]; jd=[r["label"] for r in out]
        L=["correct","hallucination","abstain"]; kp3,cm=kappa(em,jd,L)
        mask=[(e!="abstain" and j!="abstain") for e,j in zip(em,jd)]
        em2=[e for e,m in zip(em,mask) if m]; jd2=[j for j,m in zip(jd,mask) if m]
        kp2,_=kappa(em2,jd2,["correct","hallucination"])
        fH_C=sum(1 for e,j in zip(em,jd) if e=="hallucination" and j=="correct")
        fC_H=sum(1 for e,j in zip(em,jd) if e=="correct" and j=="hallucination")
        agree=sum(1 for e,j in zip(em,jd) if e==j)/max(1,len(em))
        summary[tag]=dict(n=len(out),agreement=agree,kappa3=kp3,kappa2_corr_halluc=kp2,
                          em_halluc_judge_correct=fH_C,em_correct_judge_halluc=fC_H,
                          confusion={f"em_{a}->judge_{b}":int(cm[i,k]) for i,a in enumerate(L) for k,b in enumerate(L)})
        print(f"== {tag}: agree={agree:.3f} kappa3={kp3:.3f} kappa2(c/h)={kp2:.3f} "
              f"EMhalluc->judgeCorrect={fH_C} EMcorrect->judgeHalluc={fC_H}",flush=True)
        dis=[dict(qid=r["qid"],cond=r["cond"],question=r["question"],gold=r["gold"],answer=r["answer"],
                  label_em=r["label_em"],label_judge=r["label"]) for r in out if r["label_em"]!=r["label"]]
        json.dump(dis,open(f"{WORK}/disagreements_{tag}.json","w"))
    json.dump(summary,open(f"{WORK}/judge_summary.json","w"),indent=1)
    print("\nSUMMARY\n"+json.dumps(summary,indent=1),flush=True)
    print("DONE. judged_*.json + disagreements_*.json + judge_summary.json in",WORK,flush=True)


In [ ]:
import importlib, judge_lean; importlib.reload(judge_lean)
judge_lean.run()

## After - download into pilot_v2/data/
`results_natural_bm25_qwen2_5_7b_instruct.json`, `results_natural_dense_qwen2_5_7b_instruct.json`,
all `judged_*.json`, and `judge_summary.json`. Send judge_summary.json too.